# 01 — Full-Text, Vector, and Hybrid Search on the Same Query

**Module 1.** Determine whether and how Azure AI Search meets the retrieval
requirements of an enterprise generative AI solution.

**This notebook is a demonstration, not a build-along.** It runs against a
finished index so you can see what you're building toward. You build that
index yourself in Module 2, and it must exist before you run this notebook.


## Setup — run once per kernel session

In [ ]:
# Connects to the index Module 2 built. Run once per kernel session.
import re
import sys
sys.path.insert(0, "..")

from azure.identity import DefaultAzureCredential
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizableTextQuery

from config import load_config

config = load_config()
credential = DefaultAzureCredential()
search_client = SearchClient(config.search_endpoint, config.search_index_name, credential)

print("Querying index:", config.search_index_name)
# Confirms which index you're querying. If this doesn't match .env, stop.


In [ ]:
# Two helpers the cells below reuse. No output here.
def top_hits(query, state_filter=None, k=3):
    """Hybrid + semantic — the best retrieval this index can do. Returns the
    top k, not just the top 1: a single ~2000-character page chunk often
    bundles several unrelated facts (a page of road-sign definitions, say),
    so BM25/vector similarity alone doesn't reliably rank the one sentence
    that answers a pinpoint question first. Rather than quietly picking
    result 1 and risking an off-topic passage, this shows the top few so
    you can see which one actually answers the question."""
    return list(search_client.search(
        search_text=query,
        vector_queries=[VectorizableTextQuery(text=query, k_nearest_neighbors=k, fields="vector")],
        query_type="semantic",
        semantic_configuration_name="semantic-config",
        filter=f"state eq '{state_filter}'" if state_filter else None,
        top=k,
        select=["state", "title", "chunk"],
    ))


GARBLED_CHAR = "\ufffd"  # U+FFFD REPLACEMENT CHARACTER — PDF-encoding noise, not content


def show_results(hits, label="", answer_pattern=r"\d+\s*mph", preview_len=220):
    """Flags each result with a rule, not a judgment call: GARBLED means
    PDF-encoding noise (many U+FFFD characters) — skip this one. ANSWER
    means the chunk matches answer_pattern (a number next to "mph", by
    default) — this is the one to read. Neither flag changes the order
    results print in; it only labels the same ranking the index returned.

    Both checks run on the same preview text that gets printed, not the
    full ~2000-character chunk behind it — a chunk can carry garbled text
    (a page footer, say) well past what's shown, and flagging on the full
    chunk falsely marks a clean-looking preview as GARBLED."""
    if not hits:
        print(f"{label}: no results")
        return
    print(label)
    for hit in hits:
        full = hit["chunk"].strip().replace("\n", " ")
        preview = full[:preview_len]
        reranker = hit.get("@search.reranker_score")
        if preview.count(GARBLED_CHAR) > 3:
            flag = "  <-- GARBLED, do not read"
        elif re.search(answer_pattern, preview, re.IGNORECASE):
            flag = "  <-- READ THIS ONE"
        else:
            flag = ""
        print(f"  [{hit['state']} — {hit['title']}, reranker={reranker:.2f}]{flag}")
        print(f"    {preview}...")
    print()


def mode_table(query, state_filter=None, k=3):
    flt = f"state eq '{state_filter}'" if state_filter else None
    keyword = list(search_client.search(search_text=query, filter=flt, top=k, select=["state", "title"]))
    vector = list(search_client.search(
        search_text=None,
        vector_queries=[VectorizableTextQuery(text=query, k_nearest_neighbors=k, fields="vector")],
        filter=flt, top=k, select=["state", "title"],
    ))
    hybrid = list(search_client.search(
        search_text=query,
        vector_queries=[VectorizableTextQuery(text=query, k_nearest_neighbors=k, fields="vector")],
        filter=flt, top=k, select=["state", "title"],
    ))
    rows = max(len(keyword), len(vector), len(hybrid))
    header = f"{'#':<3}{'keyword':<28}{'vector':<28}{'hybrid':<28}"
    print(header)
    print("-" * len(header))
    for i in range(rows):
        def cell(lst):
            if i >= len(lst):
                return ""
            r = lst[i]
            return f"{r['state']} ({r['@search.score']:.2f})"
        print(f"{i+1:<3}{cell(keyword):<28}{cell(vector):<28}{cell(hybrid):<28}")


---
## Clip 1 (3 min) — The Retrieval Layer and Why Enterprises Need One

No cold-model comparison — a capable model may well know a school-zone
speed limit already, and that isn't the argument. The argument: you cannot
tell where its answer came from, cannot keep it current as your documents
change, and cannot confine it to content your organization approved.
Retrieval gives you all three, regardless of what the model already knows.

**Citation note:** the chunking in Module 2 doesn't preserve the source
PDF's original page numbers (that needs a layout-aware skill, out of scope
here) — cite by state + document instead. The
quoted passage is verbatim from the index, so a learner can still verify it
in three seconds without a page number.

**This prints the top 3 matches, not just 1.** A single
~2000-character page chunk can bundle several unrelated facts (a page of
road-sign definitions, say), so similarity search alone doesn't always rank
the one sentence that answers a pinpoint question first. Note which
rank your target fact lands on for each state, rather than assuming
result 1 is always it.
(This is exactly the problem Module 2 Clip 3's filtering and semantic
ranking exist to reduce — it's already wired in here.)

Two runs of the same query, same code, only the filter changes.


In [ ]:
# I'm running against a finished index so you
# can see what we're building toward. You'll build this yourself in the
# next module.
# I won't compare this to a model with no data. The model may
# already know this answer — that's not the point. The point is you can
# see exactly where this answer came from.
hero_question = "What's the speed limit in a school zone?"

unfiltered = top_hits(hero_question)
show_results(unfiltered, "Run 1 — unfiltered")

# After running: the result marked "<-- READ THIS ONE" answers the
# question. Ignore any marked "<-- GARBLED". Note its state — the next
# cell's filter must use a different one.


In [ ]:
# Before running: confirm "florida" differs from whichever state won
# unfiltered above — if not, change it to another state, e.g. "wisconsin".
# State values are lowercase, hyphenated blob folder names.
filtered = top_hits(hero_question, state_filter="florida")
show_results(filtered, "Run 2 — filter='state eq \'florida\''")

# After running: same rule — read the result marked "<-- READ THIS ONE".
# Same question, same code — the answer is a function of the
# corpus, not the model.
# Swap these manuals for your policy set and nothing else in this
# notebook changes.


---
## Clip 2 (4 min) — Full-Text, Vector, and Hybrid Search on the Same Query

One function, three calls: keyword-only (`search_text=`), vector-only
(`vector_queries=`), and both together (hybrid). Two queries chosen so each
mode's failure mode is visible on screen.


In [ ]:
# TLSAE appears in exactly one of five manuals (Florida) — the exact-term
# case, where keyword search should dominate.
print("Query A — exact identifier, full-text should win:")
mode_table("What is TLSAE?")

# After running, check the "keyword" column is entirely Florida.
# Keyword search wins here — TLSAE is an exact acronym, and it goes
# straight to the one manual that uses it.


In [ ]:
# The manuals call this "following distance" — a phrase this question
# never uses, so keyword search has to guess from partial word matches.
print("Query B — paraphrase absent from the corpus (manuals say 'following distance'), vector should win:")
mode_table("How much space should I leave in front of me?")

# After running, compare the top keyword result to the top vector result —
# they should show different states.
# Vector search wins here, because the manuals never say it this
# way — they say 'following distance' instead.
# Full-text nails exact terms — acronyms, codes, IDs — but struggles
# when your wording differs from the document's. Vector search handles
# the paraphrase, but can lose rare identifiers. Hybrid fuses both, which
# is why it's the default.


---
## Clip 3 (3 min) — Getting Data In, and Where the Service Should Live

**No notebook code in this clip.** It happens in the Azure portal and on a
decision slide, because the objective is to *identify* connectors and
*compare* deployment models, not build either.

**Portal:** Azure AI Search service → **Import data** → look at the
connector list — Blob Storage, Azure SQL, Cosmos DB, SharePoint — and what
each is suited to. Don't configure one.

**Decision slide:** standalone Azure AI Search — reuse across apps, own the
lifecycle, full API control — versus inside a Microsoft Foundry project —
one project scope, wired to models, faster start.


---
## Done

Module 1 covers all three enabling objectives for Terminal Objective 1
using nothing but the index Module 2 already built. Next up: Module 2
shows how the retrieval layer underneath it works.
